# A Quick Recap for the each part of the AI agents field


##  LangChain vs LangGraph State & Component Comparison

This document summarizes the key concepts discussed: **LangChain**, **LangGraph**, **AgentState**, **TypedDict State**, **dataclass Context**, and **Pydantic BaseModel**.

---

### 1. High-Level Architecture

| Layer | Purpose | Typical Components | Who Owns It |
|------|---------|-------------------|-------------|
| Capability Layer | Provides AI functionality | LLMs, Tools, Prompts | LangChain |
| Execution Layer | Controls flow of execution | Graph nodes, edges, reducers | LangGraph |
| Agent Memory | Stores agent reasoning state | AgentState | LangChain |
| Workflow State | Data passed between steps | TypedDict State | LangGraph |
| Runtime Context | External configuration | dataclass context | Application |
| Data Contracts | Validate structured input/output | Pydantic BaseModel | Tools / APIs |

---

### 2. LangChain vs LangGraph

| Feature | LangChain | LangGraph |
|-------|-----------|-----------|
| Primary Role | AI capability framework | Execution orchestration |
| Core Concept | Agents | Graph workflows |
| Control Flow | Implicit agent loop | Explicit graph |
| State Management | Agent memory | Workflow state |
| Parallel Execution | Limited | Native |
| Determinism | Lower | High |
| Production Workflows | Harder to manage | Designed for production |
| Tooling | LLMs, prompts, tools | Nodes, edges, reducers |
| Interrupt Support | Agent-level | Graph-level |
| Checkpointing | Limited | Built-in |

---

### 3. AgentState vs TypedDict State

| Aspect | AgentState | TypedDict State |
|------|-------------|----------------|
| Library | LangChain | LangGraph |
| Purpose | Agent internal memory | Workflow shared state |
| Structure | Python class | Dictionary schema |
| Mutation Style | Mutable object | Functional updates |
| Scope | Single agent | Entire workflow |
| Supports Reducers | No | Yes |
| Parallel Updates | No | Yes |
| Context Growth | Continuous | Controlled |
| Used By | Tools, middleware | Graph nodes |
| Control Flow | Agent loop | Graph edges |

---

### 4. dataclass Context vs AgentState

| Aspect | dataclass Context | AgentState |
|------|------------------|-----------|
| Purpose | Runtime configuration | Agent memory |
| Mutability | Typically static | Mutable |
| Lifecycle | Provided at invocation | Changes during execution |
| Storage | Outside the agent | Inside the agent |
| Example Data | API keys, user info | authentication flags |
| Persistence | No | Optional |
| Context Window Impact | None | Yes |


### 5. TypedDict vs BaseModel

| Aspect | TypedDict | BaseModel |
|------|------------------|-----------|
| Library | Python typing | Pydantic |
| Purpose | Describe dictionary structure | Validate structured data |
| Runtime Validation | No | Yes |
| Used In | Graph state |Tools / APIs |
| Serialization | Manual | Automatic |
| Strict Typing | Static only | Runtime enforcedOptional |
| Ideal For | Workflow data flow | API schemas  |



### 6. State Evolution Comparison

| Property        | AgentState          | TypedDict         |
| --------------- | ------------------- | ----------------- |
| Memory Pattern  | Accumulating memory | Selective updates |
| Context Size    | Grows continuously  | Controlled        |
| Data Ownership  | Agent               | Workflow          |
| Update Method   | Attribute mutation  | Return dictionary |
| Parallel Safety | No                  | Yes               |


### 7. Mental Model Summary

| Concept           | Think of it as                       |
| ----------------- | ------------------------------------ |
| LangChain         | AI capability toolkit                |
| LangGraph         | Workflow execution engine            |
| AgentState        | What the agent remembers             |
| TypedDict State   | Data flowing through the workflow    |
| dataclass Context | Configuration provided to the system |
| BaseModel         | Contract enforcing structured data   |


### 8. Final Simplified Architecture Diagram
```text
Application
   |
   |-- Context (dataclass)
   |
LangGraph Workflow
   |
   |-- State (TypedDict)
   |
   |-- Node
        |
        |-- LangChain Agent
                |
                |-- AgentState
                |-- Tools
                |-- LLM
```

## Rag From Scratch: Query Transformations/Transilation

![alt text](<../../assets/Query Transilation.png>)

### Environment Initialization

Loads environment variables from `.env` and configures:
- **LangSmith tracing** — enables observability for all LangChain runs.
- **Mistral API key** — authenticates LLM and embedding calls.
- **HuggingFace token** and **User-Agent header** for web loading.

Warnings are suppressed for cleaner output.

In [ ]:
import warnings
import os 
from dotenv import load_dotenv

# Suppress all warnings for cleaner notebook output
warnings.filterwarnings("ignore")

# Load environment variables from the .env file into the process
load_dotenv()

try: 
    # Map .env variables to the keys LangChain/LangSmith expects at runtime
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")   # Enable LangSmith tracing
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")         # LangSmith authentication key
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")         # LangSmith project name for grouping traces
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")       # LangSmith API endpoint URL
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")             # Mistral AI LLM/embedding API key
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")                           # HuggingFace access token
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0"                           # Required User-Agent header for WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Part 5: Multi Query

Generates **multiple reformulations** of a user question to overcome limitations of single-query similarity search. Each reformulation retrieves different relevant chunks, and the union of results provides broader coverage.

**Flow**: User question → LLM generates 5 alternative questions → each query hits the retriever → results are deduplicated via unique union.

![alt text](../../assets/MultiQuery.png)

#### Imports & Indexing

Imports core LangChain components (`WebBaseLoader`, `RecursiveCharacterTextSplitter`, `Chroma`, `StrOutputParser`, `RunnablePassthrough`, `MistralAIEmbeddings`). Sets up path resolution using `pathlib` to locate the project root and the `db_blog` ChromaDB directory.

In [ ]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter       # Token-aware text chunking
from langchain_community.document_loaders import WebBaseLoader            # Fetch web pages as Documents
from langchain_community.vectorstores import Chroma                       # ChromaDB vector store integration
from langchain_core.output_parsers import StrOutputParser                 # Extracts raw string from LLM response
from langchain_core.runnables import RunnablePassthrough                  # Passes input through unchanged in LCEL chains
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings        # Mistral LLM and embedding models
from pathlib import Path
from langsmith import Client
client = Client()  # LangSmith client for pulling prompts and tracing

# Resolve the project root: .parent[0] = part_5_9/, .parent[1] = src/, .parents[1] = rag_tutorial/
ROOT_DIR = Path.cwd().parents[1]

# Path to the shared ChromaDB database directory at the project root
DB_DIR = ROOT_DIR / "db_blog"

#### Load, Split & Store Documents

Fetches the blog post, splits it into 300-token chunks (50 overlap) using tiktoken, embeds with `mistral-embed`, and stores in the `"blog_posts"` ChromaDB collection. Creates a retriever returning the top 3 most relevant documents (`k=3`).

In [ ]:
# ==========================
# INDEXING: Load → Split → Embed → Store
# ==========================

# Load the blog post HTML, filtering only article-relevant CSS classes
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")  # Keep only main content
        )
    ),
)
blog_docs = loader.load()  # Returns a list of Document objects

# Split documents using tiktoken-based token counting (not raw character count)
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,       # Max 300 tokens per chunk
    chunk_overlap=50,     # 50-token overlap to preserve context at boundaries
)
splitted_docs = splitter.split_documents(blog_docs)

# Embed and persist chunks in ChromaDB
embeddings = MistralAIEmbeddings(model = "mistral-embed")
vectorstore = Chroma.from_documents(
    documents = splitted_docs,
    embedding = embeddings,
    collection_name = "blog_posts",      # Collection name inside ChromaDB
    persist_directory = str(DB_DIR)      # Persist to the shared db_blog directory
)

# Create a retriever that returns the top-3 most relevant chunks
# k controls how many documents appear in the LangSmith trace view
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

#### Multi-Query Prompt & Chain

Defines a prompt that instructs the LLM to generate **5 alternative versions** of the user's question. The chain pipes the prompt through `mistral-small-latest`, parses the output string, and splits it into a list of individual queries (filtering empty lines).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Multi-Query: Generate 5 alternative perspectives of the user's question
# This helps overcome limitations of single-query distance-based similarity search
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Do not include any preamble.
Original question: {question}"""

# Build the prompt template from the string
prompt_perspectives = ChatPromptTemplate.from_template(template)

# Chain: prompt → LLM → parse string → split into list of non-empty query strings
generate_query_chain = (
    prompt_perspectives 
    | ChatMistralAI(model = "mistral-small-latest", temperature=0)
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])  # Split output lines, filter blanks
)

#### Retrieve & Deduplicate (Unique Union)

- `get_unique_union()` — Flattens the list-of-lists from `.map()`, serializes each `Document` to JSON, deduplicates via `set`, and deserializes back.
- The retrieval chain: generate queries → `retriever.map()` (runs retriever on each query in parallel) → unique union.
- Returns the total number of unique retrieved documents.

In [ ]:
from langchain_core.load import loads, dumps

def get_unique_union(documents: list[list]): 
    """Deduplicate documents retrieved from multiple queries.
    
    Flattens list-of-lists, serializes each Document to JSON for set-based
    deduplication, then deserializes back to Document objects.
    """
    # Flatten nested lists and serialize each Document to a JSON string
    flatten_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Deduplicate using set (identical JSON strings collapse to one)
    unique_docs = list(set(flatten_docs))
    # Deserialize back to Document objects
    return [loads(docs) for docs in unique_docs]


# Test the multi-query retrieval pipeline
question = "What is task decomposition for LLM agents?"

# Chain: generate queries → run retriever on each query in parallel (.map()) → deduplicate
retrieval_chain = (
    generate_query_chain | retriever.map() | get_unique_union)

docs = retrieval_chain.invoke({"question": question})

len(docs)  # Total unique documents retrieved across all query perspectives

3

#### Preview Retrieved Documents

Iterates over the retrieved documents and prints each one's source URL and a 400-character content preview. Useful for verifying that the multi-query retrieval returned relevant and diverse chunks.

In [ ]:
# Preview the retrieved documents to verify relevance and diversity
for i, doc in enumerate(docs):
    print(f"\n{'='*40}")
    print(f"📄 DOCUMENT {i+1}")
    print(f"{'='*40}")
    
    # Safely extract the source URL; .get() avoids KeyError if 'source' is missing
    source = doc.metadata.get('source', 'Unknown Source')
    print(f"🔗 Source: {source}\n")
    
    # Print the first 400 characters as a quick content preview
    print("📝 Content Preview:")
    print(f"{doc.page_content[:400]}...")


📄 DOCUMENT 1
🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/

📝 Content Preview:
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays an...

📄 DOCUMENT 2
🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/

📝 Content Preview:
(2) Model selection: LLM distributes the tasks to expert models, where the request is framed as a multiple-choice question. LLM is presented with a list of models to choose from. Due to the limited context length, task type based filtration is needed.
Instruction:

Given the user request and the call command, the AI assistant helps the user to select a suitable model from a list of models to proce..

#### Multi-Query RAG Chain

Assembles the final RAG chain for Part 5:
- **Context** comes from the multi-query retrieval chain (unique union of results).
- **Question** is extracted via `itemgetter`.
- Piped through a prompt template → `mistral-small-latest` → `StrOutputParser` to produce the final answer.

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_mistralai import ChatMistralAI

# Define the RAG prompt template with context and question placeholders
template = """Answer the following question based on this context: 

Context: {context}


Question: {question}
""" 

prompt = ChatPromptTemplate.from_template(template)

# Initialize LLM with deterministic output
llm = ChatMistralAI(model = "mistral-small-latest", temperature = 0)

# Assemble the final Multi-Query RAG chain:
#   - "context": multi-query retrieval chain (unique union of docs from multiple queries)
#   - "question": extracted from the input dict via itemgetter
#   - Piped through prompt → LLM → string parser
final_rag_chain = (
    {"context": retrieval_chain, 
    "question": itemgetter("question")} 
    | prompt 
    | llm
    | StrOutputParser()
)

# Invoke with a test question
docs = final_rag_chain.invoke({"question": "What is the main topic of the blog post?"})

### Part 6: RAG Fusion

An improvement over Multi-Query that adds **Reciprocal Rank Fusion (RRF)** scoring. Instead of simple deduplication, documents are scored based on their rank across multiple query results, producing a better-ordered final list.

**Flow**: User question → LLM generates 4 related queries → each query retrieves docs → RRF re-ranks all results by fused score → top docs feed the RAG chain.

![alt text](<../../assets/Rag Fusion.png>)

#### RAG Fusion Prompt

Defines a prompt that asks the LLM to generate **4 related search queries** from a single input question. The output is parsed and split into a list of query strings.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# RAG Fusion: Generate 4 related search queries from a single input question
# Unlike Multi-Query (which rephrases), RAG Fusion generates related queries
# that explore different facets of the topic
template = """You are a helpful assistant that generates multiple search queries based on a single input query without any preamble. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

#### Query Generation Chain

Chains the RAG Fusion prompt with `mistral-small-latest`, parses the string output, and splits it into individual queries (filtering blank lines).

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(model = "mistral-small-latest")

# Chain: RAG Fusion prompt → LLM → parse string → split into list of query strings
generated_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])  # Split on newlines, filter blanks
)

#### Reciprocal Rank Fusion (RRF) & Retrieval

`reciprocal_rank_fusion()` re-ranks documents from multiple retrieval lists using the formula $\text{score}(d) = \sum \frac{1}{\text{rank} + k}$ (default $k=60$). Documents appearing at higher ranks across more queries get higher fused scores.

The retrieval chain: generated queries → `retriever.map()` → RRF → sorted results with scores.

In [ ]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """Re-rank documents using Reciprocal Rank Fusion (RRF).
    
    For each document, computes: score(d) = sum(1 / (rank + k)) across all query result lists.
    Documents appearing at higher ranks in more lists get higher fused scores.
    
    Args:
        results: List of ranked document lists (one per query).
        k: Smoothing constant (default=60, standard RRF value).
    
    Returns:
        List of (Document, score) tuples sorted by descending fused score.
    """
    # Dictionary mapping serialized document → cumulative RRF score
    fused_scores = {}

    # Iterate through each retrieval result list
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)  # Serialize Document to JSON string for use as dict key
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0  # Initialize score for new documents
            # Apply the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort documents by fused score in descending order (best first)
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    return reranked_results

# Chain: generate queries → retrieve docs for each (.map()) → RRF re-ranking
retrieval_chain_rag_fusion = (
    generated_queries
    | retriever.map() 
    | reciprocal_rank_fusion
)

docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)  # Number of unique re-ranked documents

2

#### RAG Fusion Final Chain

Assembles the final RAG chain using the RRF-ranked context. The fused results feed into a prompt template with `{context}` and `{question}`, piped through the LLM and output parser to produce the answer.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Define the RAG prompt template
template = """Answer the following question based on this context: 
{context}


Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# Initialize LLM
llm = ChatMistralAI(model="mistral-small-latest")

# Test question
question = "What is task decomposition for LLM agents?"

# Assemble the final RAG Fusion chain:
#   - "context": RRF-ranked docs from the fusion retrieval chain
#   - "question": extracted via itemgetter from the input dict
#   - Piped through prompt → LLM → string output parser
final_fusion_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
    "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_fusion_rag_chain.invoke({"question": question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down a complex task into smaller, more manageable sub-tasks or steps. This technique helps the agent systematically approach and solve complicated tasks by simplifying them into a sequence of easier-to-handle components.\n\n### Key Methods of Task Decomposition:\n1. **Chain of Thought (CoT)**:\n   - The model is prompted to "think step by step," decomposing a complex task into smaller, sequential steps.\n   - This enhances the model\'s reasoning by leveraging more computational effort during inference.\n\n2. **Tree of Thoughts (ToT)**:\n   - Extends CoT by exploring multiple reasoning paths at each step, creating a tree-like structure of possible solutions.\n   - The search can be done using breadth-first search (BFS) or depth-first search (DFS), with evaluations via a classifier or majority voting.\n\n3. **Prompting Techniques**:\n   - Simple prompts like *"Steps for XYZ. 1."* or *"What are the

### Part 7: Decomposition

Breaks a complex question into **smaller sub-questions** that can each be answered independently. Two strategies are demonstrated:
1. **Recursive** — Each sub-question's answer becomes context for the next, building up a chain of reasoning.
2. **Individual** — Each sub-question is answered in isolation, then all Q+A pairs are synthesized into a final answer.

#### Reconnect to VectorDB

Reopens the existing `"blog_posts"` ChromaDB collection using `pathlib` for cross-platform path resolution. Verifies the connection by printing the item count. Creates a retriever for use in subsequent cells.

In [ ]:
from langchain_chroma import Chroma                                      # Updated Chroma import (langchain-chroma package)
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from pathlib import Path

# Initialize the same embedding model used during indexing
embeddings = MistralAIEmbeddings(model="mistral-embed")

# Resolve path to the shared ChromaDB database at the project root
ROOT_DIR = Path.cwd().parents[1]       # Navigate up from src/part_5_9/ to rag_tutorial/
DB_DIR = ROOT_DIR / "db_blog"          # Shared database directory

try: 
    # Reconnect to the existing ChromaDB collection (no re-indexing needed)
    vectorstore = Chroma(
        embedding_function = embeddings,
        collection_name = "blog_posts",        # Must match the collection created during indexing
        persist_directory = str(DB_DIR)         # Path to the persisted database
    )
    
    # Verify the collection has data
    count = vectorstore._collection.count()
    print(f"Total items in 'blog_posts': {count}")
    if count > 0:
        print("Database opened successfully with existing data.")
except ConnectionError as e: 
    print("Database is empty. Check your path or if data was previously saved. The DB will be Initialized.")

# Create a retriever for downstream chains
retriever = vectorstore.as_retriever()

Total items in 'blog_posts': 350
Database opened successfully with existing data.


#### Generate Sub-Questions

Prompts the LLM to decompose a complex question into **3 sub-questions** that can be answered independently. The chain parses the output and splits it into a list of sub-question strings.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Decomposition prompt: break a complex question into 3 independent sub-questions
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation without preambles. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

# Initialize LLM with deterministic output
llm = ChatMistralAI(model = "mistral-small-latest", temperature=0)

# Chain: prompt → LLM → parse string → split into list of sub-question strings
generate_query_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])  # Split on newlines, filter blanks
)

# Generate sub-questions for the test question
question = "What are the main components of an LLM-powered autonomous agent system?"
queries = generate_query_decomposition.invoke({"question": question})

print("\n".join(queries))  # Display the generated sub-questions

1. **"What are the core components of an LLM-powered autonomous agent system?"**
2. **"What are the key modules required for an autonomous agent using large language models?"**
3. **"How do LLM-based autonomous agents integrate different system components?"**


#### Answer Recursively

Each sub-question is answered sequentially: the answer to sub-question N becomes part of the context for sub-question N+1. This builds cumulative understanding.

- **Prompt** includes the current question, all previous Q+A pairs, and retrieved context.
- The loop iterates over sub-questions, accumulating formatted Q+A pairs as background context.

In [ ]:
# Recursive decomposition prompt: answers build on previous Q+A pairs
# {question}   — current sub-question to answer
# {q_a_pairs}  — accumulated background from prior sub-questions
# {context}    — retrieved documents relevant to this sub-question
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

#### Recursive RAG Loop

Iterates through each sub-question: retrieves relevant docs, combines them with accumulated Q+A pairs as background context, generates an answer, and appends the new Q+A pair to the accumulator. Each iteration enriches the context for the next sub-question.

In [ ]:
from operator import itemgetter

def format_qa_pair(question, answer): 
    """Format a single question-answer pair as a readable string."""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# Initialize LLM
llm = ChatMistralAI(model = "mistral-small-latest")

# Accumulator for all previous Q+A pairs (builds up each iteration)
q_a_pairs = ""

# Recursive loop: each sub-question's answer enriches context for the next
for q in queries:
    # Build a RAG chain for this sub-question:
    #   - "context": retrieve docs relevant to the current sub-question
    #   - "question": the current sub-question text
    #   - "q_a_pairs": accumulated background from all prior sub-questions
    rag_chain = (
        {
        "context": itemgetter("question") | retriever,   # Retrieve docs for this sub-question
        "question": itemgetter("question"),               # Pass the sub-question through
        "q_a_pairs": itemgetter("q_a_pairs")              # Pass accumulated Q+A context
        } 
    | decomposition_prompt
    | llm
    | StrOutputParser()
    )
    
    # Generate the answer for this sub-question
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    
    # Format the new Q+A pair and append to the accumulator
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair
    
    print(q, ":\n", answer, "\n ")

1. **"What are the core components of an LLM-powered autonomous agent system?"** :
 The core components of an LLM-powered autonomous agent system, as described in the provided context, include:

1. **LLM (Large Language Model) as the Core Controller**: The LLM serves as the "brain" of the agent, enabling it to process information, generate responses, and make decisions.

2. **Planning**:
   - **Subgoal and Decomposition**: The agent breaks down complex tasks into smaller, manageable subgoals to handle them efficiently.
   - **Reflection and Refinement**: The agent can self-critique and refine its actions based on past performance, improving future steps and outcomes.

3. **Memory**: The agent retains information from past interactions or tasks, allowing it to learn and adapt over time.

These components work together to enable the agent to function as a general problem solver, extending beyond simple text generation to more complex, autonomous tasks. Examples of such systems include Au

#### Answer Individually

Each sub-question is answered **independently** using its own retrieved context (no cross-referencing previous answers).

- `retrieve_and_rag()` generates sub-questions, retrieves docs for each, and runs a separate RAG chain per sub-question.
- All individual answers are then formatted as Q+A pairs and synthesized into a single comprehensive response.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langsmith import Client
client = Client()

# Pull the community RAG prompt from LangSmith Hub
prompt_rag = client.pull_prompt("rlm/rag-prompt")

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """Answer each sub-question independently using its own retrieved context.
    
    Unlike the recursive approach, each sub-question is answered in isolation
    (no cross-referencing previous answers).
    
    Args:
        question: The original complex question to decompose.
        prompt_rag: The RAG prompt template.
        sub_question_generator_chain: Chain that generates sub-questions.
    
    Returns:
        Tuple of (answers_list, sub_questions_list).
    """
    # Decompose the question into sub-questions
    sub_questions = sub_question_generator_chain.invoke({"question": question})
    
    # Collect answers for each sub-question
    rag_results = []
    
    for sub_question in sub_questions: 
        # Retrieve documents specific to this sub-question
        retrieved_docs = retriever.invoke(sub_question)
        
        # Run an independent RAG chain for this sub-question
        answer = (
            prompt_rag 
            | llm 
            | StrOutputParser()
        ).invoke({
                "question": sub_question,
                "context": retrieved_docs})
        
        # Append the answer to results
        rag_results.append(answer)
    
    return rag_results, sub_questions

# Generate individual answers for each sub-question
answers, questions = retrieve_and_rag(question, prompt_rag, generate_query_decomposition)

#### Synthesize Individual Answers

Formats all sub-question/answer pairs into a structured string, then passes them to a final synthesis prompt. The LLM combines all partial answers into one coherent response to the original question.

In [ ]:
def format_qa_pairs(questions, answers):
    """Format multiple Q+A pairs into a numbered, readable string.
    
    Args:
        questions: List of sub-question strings.
        answers: List of corresponding answer strings.
    
    Returns:
        Formatted string with numbered Q+A pairs.
    """
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start = 1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

# Format all individual Q+A pairs into a single context string
context = format_qa_pairs(questions, answers)

# Synthesis prompt: combine all sub-question answers into one coherent response
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# Final synthesis chain: prompt → LLM → string output
final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

# Synthesize a single comprehensive answer from all sub-question results
final_rag_chain.invoke({"context": context, "question": question})

'The main components of an **LLM-powered autonomous agent system** include:\n\n1. **Large Language Model (LLM)** – Acts as the core "brain" of the system, processing inputs, generating responses, and orchestrating decision-making.\n2. **Planning Module** – Enables task breakdown through **subgoal decomposition** and **reflection/refinement**, allowing the agent to handle complex tasks systematically.\n3. **Memory** – Stores past actions, experiences, and learned insights, enabling the agent to improve over time by learning from mistakes and adapting strategies.\n\nThese components work together to allow the agent to autonomously decompose tasks, execute actions, reflect on outcomes, and refine its approach for better performance. The LLM serves as the central controller, while planning and memory enhance its ability to solve problems efficiently and iteratively.'

### Part 8: Step Back Prompting

Generates a **more generic "step-back" question** from the user's specific query. The broader question retrieves higher-level context that complements the specific retrieval. Both contexts are combined for a more comprehensive answer.

**Flow**: User question → LLM generates step-back question → retrieve context for both → combine contexts → generate answer.

#### Few-Shot Step-Back Prompt

Uses `FewShotChatMessagePromptTemplate` to teach the LLM how to "step back" via 2 examples:
- *"Could the members of The Police perform lawful arrests?"* → *"What can the members of The Police do?"*
- *"Jan Sindel's was born in what country?"* → *"What is Jan Sindel's personal history?"*

The chain generates a broader, more generic version of the user's question using `mistral-small-latest` (temperature=0).

In [ ]:
# Few-Shot Step-Back Prompting
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Two examples teaching the LLM how to "step back" to a more generic question
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel's was born in what country?",
        "output": "what is Jan Sindel's personal history?",
    },
]

# Wrap examples as human/ai message pairs for few-shot prompting
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# Build the few-shot prompt object from the examples
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_prompt,
    examples = examples
)

# Full prompt: system instruction + few-shot examples + new user question
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        few_shot_prompt,         # Inject the few-shot examples
        ("user", "{question}"),  # The user's specific question
    ]
)

# Initialize LLM with deterministic output
llm = ChatMistralAI(model = "mistral-small-latest", temperature=0)

# Chain: prompt → LLM → parse to string (the step-back question)
generate_queries_step_back = (
    prompt
    | llm
    | StrOutputParser()
)

# Test: generate a step-back question from a specific query
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

'What is task decomposition?'

#### Step-Back RAG Chain

Builds a dual-retrieval chain:
- **`normal_context`** — Retrieves docs using the original question via `RunnableLambda`.
- **`step_back_context`** — Retrieves docs using the step-back question generated by the few-shot chain.
- Both contexts are passed to a response prompt that synthesizes a comprehensive, non-contradictory answer.

In [ ]:
# Response prompt: instructs the LLM to use both normal and step-back context
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

# Dual-retrieval chain:
#   - "normal_context": retrieve docs using the original question directly
#   - "step_back_context": generate a step-back question first, then retrieve docs
#   - "question": pass the original question through for the final answer
chain = (
    {
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,     # Direct retrieval
        "step_back_context": generate_queries_step_back | retriever,                # Step-back retrieval
        "question": lambda x: x["question"],                                        # Pass question through
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

chain.invoke({"question": question})

'Task decomposition is a critical component in the planning phase of LLM-powered autonomous agents, enabling them to break down complex tasks into smaller, more manageable subgoals. This process enhances the agent\'s ability to handle intricate problems systematically. Here’s a detailed breakdown of task decomposition for LLM agents:\n\n### **1. Definition and Purpose**\nTask decomposition involves dividing a large, complex task into smaller, sequential or parallel subgoals. This allows the agent to:\n- **Simplify problem-solving**: Complex tasks are easier to manage when broken into smaller steps.\n- **Improve efficiency**: The agent can focus on one subgoal at a time, reducing cognitive load.\n- **Enhance adaptability**: The agent can adjust its approach based on the success or failure of individual subgoals.\n\n### **2. Methods of Task Decomposition**\nSeveral techniques are used to decompose tasks in LLM agents:\n\n#### **a. Chain of Thought (CoT)**\n- **Concept**: The agent is pro

### Part 9: HyDE (Hypothetical Document Embeddings)

Instead of embedding the raw question, the LLM first generates a **hypothetical answer document**, which is then embedded and used for retrieval. This bridges the gap between question-style and document-style embeddings.

**Flow**: User question → LLM writes a hypothetical passage → embed that passage → retrieve similar real documents → RAG with retrieved docs.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

# HyDE: Generate a hypothetical document that answers the question
# This bridges the embedding gap between question-style and document-style text
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""

prompt_HyDE = ChatPromptTemplate.from_template(template) 

# Chain: prompt → LLM generates a hypothetical passage → parse to string
general_doc_retrieval = (
    prompt_HyDE 
    | ChatMistralAI(model = "mistral-small-latest", temperature=0)
    | StrOutputParser()
)

# Generate the hypothetical document for embedding-based retrieval
question = "What is task decomposition for LLM agents?"
generated_docs = general_doc_retrieval.invoke({"question": question})

#### Generate Hypothetical Document

Prompts `mistral-small-latest` to write a scientific paper passage answering the question. This hypothetical document will be used as the search query instead of the raw question, improving embedding alignment with stored document chunks.

In [ ]:
# Preview the hypothetical document generated by HyDE
print(generated_docs)

**Task Decomposition for LLM Agents: A Systematic Approach to Complex Problem Solving**

Large Language Model (LLM) agents are increasingly employed to tackle complex, multi-step tasks that require reasoning, planning, and execution. However, directly applying an LLM to an intricate problem without structuring the workflow can lead to inefficiencies, errors, or incomplete solutions. **Task decomposition** is a critical technique that breaks down a high-level objective into smaller, manageable subtasks, enabling LLMs to process and solve problems systematically.

### **Definition and Purpose**
Task decomposition involves dissecting a complex task into a sequence of simpler, interdependent subtasks. This approach aligns with human problem-solving strategies, where breaking down a problem into smaller parts reduces cognitive load and improves accuracy. For LLM agents, decomposition enhances performance by:
1. **Reducing computational overhead** – Smaller subtasks require less context and 

#### HyDE Retrieval & RAG

- **Retrieval chain**: The hypothetical document is piped into the retriever, which embeds it and finds the most similar real chunks.
- **Final RAG chain**: The retrieved real documents and the original question feed into a prompt → LLM → `StrOutputParser` to produce the final answer.

In [ ]:
# HyDE Retrieval: embed the hypothetical document and find similar real chunks
# The generated passage is more "document-like" than the raw question,
# so it aligns better with stored document embeddings
retrieval_chain = (
    general_doc_retrieval   # Generate hypothetical doc
    | retriever             # Embed it and retrieve similar real documents
)
retrieved_docs = retrieval_chain.invoke({"question": question})
retrieved_docs  # Inspect the retrieved real documents

[Document(id='d34f7a6c-4876-4272-8da7-0b5da7e65bdc', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-f

In [ ]:
# Final HyDE RAG chain: use the retrieved real documents to answer the question
template = """Answer the following question based on this context:

    {context}

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)

# Chain: prompt (with retrieved docs as context) → LLM → string output
final_rag_chain_Hyde = (
        prompt
        | llm
        | StrOutputParser()
    )

# Invoke with the real retrieved docs and the original question
final_rag_chain_Hyde.invoke({"context": retrieved_docs, "question": question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down a complex task into smaller, more manageable sub-tasks or steps. This approach helps the agent handle complicated tasks systematically by planning and executing them in a structured manner. Here are the key aspects of task decomposition for LLM agents based on the provided context:\n\n1. **Chain of Thought (CoT)**:\n   - A standard prompting technique where the model is instructed to "think step by step."\n   - It decomposes a complex task into smaller, simpler steps, making it easier for the model to process and solve.\n   - CoT provides insight into the model\'s reasoning process by transforming a big task into multiple manageable sub-tasks.\n\n2. **Tree of Thoughts (ToT)**:\n   - An extension of CoT that explores multiple reasoning possibilities at each step.\n   - It decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree-like structu